# PaddleOCR-VL-1.6 on the hardest FinTabNet tables (Colab)

Phase-A/C specialist ceiling run. Scores **PaddleOCR-VL-1.6 (0.9B)** — a table-SOTA
document parser — over the 50 hardest FinTabNet validation tables (merged cells
guaranteed), through the repo's existing TEDS-Struct + span-recall + bootstrap-CI
pipeline. GLM-OCR is added in a second notebook once this one is proven end-to-end.

**Public data only.** FinTabNet is public S&P-500 tables, so nothing here touches the
confidential-invoice boundary — Colab is fine.

**This is a debugging run, by design.** `paddle_client._extract_table_html` guesses the
PaddleOCR-VL result-object shape (the one open item flagged in CLAUDE.md). Cell 4
inspects the raw result on a single image *before* the full 50-table pass so you can
confirm/patch the extraction there instead of at the end.

**Prereq — push the branch first.** `paddle_client.py`, `api_client.py`, `images.py`
and the `inference.py` model constants must be committed and pushed, or the clone below
won't contain them:
```
git add src/ && git commit -m "phase-A specialist clients" && git push -u origin phase-two-distillation-pipeline
```
Runtime: **GPU** (T4 is plenty for 0.9B) — Runtime ▸ Change runtime type ▸ T4 GPU.

## 1. Setup — clone repo + install PaddleOCR-VL

`paddleocr`/`paddlepaddle` are deliberately **not** in `requirements-base.txt` (kept out
so `src/` imports without them on the Mac). We add them here. Two gotchas:

- **The `paddlex[ocr]` extra is required.** Installing `paddleocr` pulls in `paddlex`
  but not its `[ocr]` optional deps, which the PaddleOCR-VL pipeline needs at
  construction time. Without them, `PaddleOCRVL()` in cell 3 raises
  `DependencyError: PaddleOCR-VL-1.6 requires additional dependencies`. The cell below
  installs it, pinned to the resolved paddlex version.
- **Version drift.** PaddleOCR-VL needs paddleocr 3.x on paddlepaddle 3.x; if
  `PaddleOCRVL` fails to *import* (rather than construct), that's the first thing to bump.

In [ ]:
BRANCH = "phase-two-distillation-pipeline"
REPO   = "https://github.com/hidrochin/qwen-vl-table-reconstruction.git"

import os, sys

# Resolve from /content every time so re-running this cell can't nest a second clone
# (.../qwen-vl-table-reconstruction/qwen-vl-table-reconstruction). Idempotent whether the
# runtime is fresh or the cell is re-run.
REPO_DIR = "/content/qwen-vl-table-reconstruction"
%cd /content
if not os.path.isdir(REPO_DIR):
    !git clone --branch $BRANCH $REPO
%cd $REPO_DIR
!git checkout $BRANCH && git pull --ff-only

# Repo runtime deps (loader uses requests; eval uses numpy). CPU-only, installs fast.
!pip install -q -r requirements-base.txt
# Specialist pipeline — the two packages held out of requirements-base.
!pip install -q -U "paddlepaddle-gpu" "paddleocr"
# `paddleocr` pulls in `paddlex` but NOT its [ocr] extra, which carries the table/layout
# runtime deps the PaddleOCR-VL pipeline needs. Without it, constructing PaddleOCRVL()
# raises `DependencyError: PaddleOCR-VL-1.6 requires additional dependencies`. Pin the
# extra to the paddlex version already installed above (unpinned can drag paddlex to an
# incompatible release).
from importlib.metadata import version as _pkg_version
_PADDLEX_VER = _pkg_version("paddlex")
!pip install -q "paddlex[ocr]=={_PADDLEX_VER}"

# Make `import src...` work from the repo root.
sys.path.insert(0, os.getcwd())
print("\ncwd:", os.getcwd())

## 2. Get the eval corpus — restore cached (fast) or build (slow)

Two ways to get `data/corpus/eval` (the 50 hardest tables + `manifest.jsonl`):

- **Fast (recommended once you have the zip):** upload the `fintabnet_eval_corpus.zip`
  you downloaded last session — Files panel (◀ left sidebar) ▸ upload, or uncomment the
  `files.upload()` line in the **restore** cell. It unzips into `data/corpus/`, and the
  **build** cell below then sees the manifest and **skips the scan entirely**.
- **Slow (first run only):** no zip present → the build cell scans FinTabNet fresh.

Both paths end with `records` loaded, so the rest of the notebook is identical.

**Rate limit (the earlier failure).** The anonymous `datasets-server` API returns
**HTTP 429** under sustained paging — that's what killed the first run at ~2,000 rows,
not a model error. Mitigations already in place: `max_scanned=5000` (a portion, not the
whole 20k split — ~2k spanning candidates by then), and the loader paces requests,
honors `Retry-After`, and waits out a 429 instead of dying. Set `HF_TOKEN` to lift the
limit further; bump `max_scanned` if you see a `wanted 50, got N` warning.

In [ ]:
# --- FAST PATH (optional): reuse last session's corpus instead of re-scanning ----------
# If fintabnet_eval_corpus.zip is in the working dir, unzip it into data/corpus/ so the
# build cell below skips the slow, rate-limited scan. Get the zip there via the Files
# panel (drag it in) or by uncommenting the upload line. If no zip is present this is a
# harmless no-op and the build cell scans fresh — so "Run all" is safe either way.
from pathlib import Path
from src.data.loader import load_manifest

CORPUS = Path("data/corpus")
ZIP = "fintabnet_eval_corpus.zip"

# from google.colab import files; files.upload()   # <- uncomment to pick the zip now

if os.path.exists(ZIP):
    !unzip -o -q "$ZIP" -d data/corpus
    n = len(load_manifest(CORPUS / "eval"))
    print(f"restored {n} tables from {ZIP} -> {CORPUS/'eval'} (build cell will skip the scan)")
else:
    print(f"no {ZIP} in {os.getcwd()} -- the build cell will scan FinTabNet fresh")

In [ ]:
from pathlib import Path
from src.data.loader import build_split, load_manifest

# OPTIONAL but recommended: an HF token lifts the datasets-server rate limit a lot,
# so the scan won't throttle. Get one at https://huggingface.co/settings/tokens
# (read scope is enough) and uncomment:
# import os; os.environ["HF_TOKEN"] = "hf_xxx"

CORPUS = Path("data/corpus")
if (CORPUS / "eval" / "manifest.jsonl").exists():
    # Corpus already on disk — freshly built earlier, or restored from the zip in the
    # cell above. Reuse it and skip the slow, rate-limited scan. Delete data/corpus (or
    # the manifest) to force a rebuild.
    records = load_manifest(CORPUS / "eval")
    corpus_was_built = False
    print(f"reusing existing corpus: {len(records)} tables -> {CORPUS/'eval'}")
else:
    records = build_split(
        split_name="eval",
        source_split="validation",
        n_target=50,
        out_dir=CORPUS,
        # Scan a portion, not the whole split. The anonymous datasets-server API 429s
        # under sustained paging; 5,000 rows already yields ~2k spanning candidates, so
        # the hardest 50 are well-covered. Raise this (or set HF_TOKEN above) if you see
        # a "wanted 50, got N" warning.
        max_scanned=5000,
    )
    corpus_was_built = True
    print(f"\nbuilt {len(records)} tables -> {CORPUS/'eval'}")

In [ ]:
# --- Save the built corpus to your machine ------------------------------------------
# The FinTabNet scan is the slow, rate-limited part. Zip data/corpus/eval (the 50 images
# + manifest.jsonl, ~3 MB) and download it so you can reuse it next session instead of
# re-scanning. The manifest stores image paths RELATIVE ("images/<uid>.jpg"), so the
# archive is portable: unzip it into data/corpus/ on any machine and load_manifest works.
#
# Only fires when the corpus was BUILT this session (corpus_was_built from the cell above)
# — if it was restored from an uploaded zip there's nothing new to save, so "Run all"
# won't nag you with a redundant download. Force a download anyway by setting force=True.
import shutil
from google.colab import files

force = False
if corpus_was_built or force:
    archive = shutil.make_archive(
        "fintabnet_eval_corpus", "zip", root_dir="data/corpus", base_dir="eval"
    )
    print(f"wrote {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")
    files.download(archive)   # triggers the browser download prompt
else:
    print("corpus was reused from cache -> nothing new to download (set force=True to override)")

# Reuse next session: upload this zip (or drag it into the Files panel) and the restore
# cell in section 2 unzips it, so the build cell skips the scan.

## 3. Load PaddleOCR-VL

First construction downloads the 0.9B weights. `PaddleTableReconstructor` wraps the
pipeline behind `predict`/`predict_many` so its output flows into the same eval path as
every other candidate.

In [4]:
from src.model.paddle_client import PaddleTableReconstructor, _extract_table_html

reconstructor = PaddleTableReconstructor()
pipeline = reconstructor._get_pipeline()   # triggers the weight download
print("PaddleOCR-VL ready")

Creating model: ('PP-DocLayoutV3', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-DocLayoutV3), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-DocLayoutV3`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

AttributeError: 'paddle.base.libpaddle.AnalysisConfig' object has no attribute 'set_optimization_level'

## 4. Debugging pass — confirm the result-object shape on ONE image

**This is the cell that de-risks the run.** `_extract_table_html` walks the pipeline
output looking for `<table>…</table>`. If it returns empty here, patch the extraction
(inspect `raw` below and adjust the attr list / traversal in `src/model/paddle_client.py`)
*before* spending time on all 50. Don't proceed to cell 5 until this prints a table.

In [ ]:
records = load_manifest(CORPUS / "eval")
sample = records[0]
print("image:", sample.image_path, "\n")

raw_result = pipeline.predict(sample.image_path)
print("type(raw_result):", type(raw_result))
print("repr (truncated):\n", repr(raw_result)[:1500], "\n")

extracted = _extract_table_html(raw_result)
print("=== extracted <table> (first 800 chars) ===")
print(extracted[:800] if extracted else "*** EMPTY — patch _extract_table_html before continuing ***")

## 5. Predict over all 50 tables

In [ ]:
image_paths = [r.image_path for r in records]
preds = reconstructor.predict_many(image_paths)
print(f"\n{len(preds)} predictions; "
      f"{sum(1 for p in preds if not p.html.strip())} empty")

## 6. Score + report

Same pipeline as the API bake-off: TEDS-Struct (structure-only), position-aware span
recall, bootstrap CIs, per-difficulty-bin breakdown, and parse failures. Remember these
specialists emit their **own HTML dialect** — read TEDS-Struct as a *ceiling reference*
("what a table-SOTA parser gets zero-shot"), and lean on **span recall** for the
customer-legible "recovered N of M merged cells" number.

In [ ]:
from src.eval.runner import evaluate_predictions, save_run
from src.model.inference import predictions_dict

results, summary = evaluate_predictions(records, predictions_dict(preds), "paddleocr-vl-1.6")
save_run(results, summary, Path("outputs/runs"))

print(f"=== PaddleOCR-VL-1.6 over {summary.n} hardest FinTabNet tables ===")
print(f"TEDS-Struct : {summary.mean_teds:.4f}  [{summary.ci_low:.3f}, {summary.ci_high:.3f}]")
print(f"span-recall : {summary.mean_span_recall:.4f}")
print(f"parse-fails : {summary.parse_failures}/{summary.n}")
if summary.by_bin:
    print("\nby difficulty bin:")
    for label, b in summary.by_bin.items():
        print(f"  {label:<7} n={b['n']:<3} TEDS {b['mean']:.4f} [{b['ci'][0]:.3f}, {b['ci'][1]:.3f}]")

NameError: name 'records' is not defined

## 7. Eyeball the hardest failures

Worst TEDS-Struct cases first — this is where dialect-vs-real-error gets sorted out.

In [ ]:
from src.eval.runner import to_cases

for c in to_cases(results, limit=3, hardest_first=True):
    print("=" * 70)
    print(f"{c.uid}  TEDS-Struct={c.score:.3f}  difficulty={c.difficulty}  spans={c.n_spanning}")
    print("  image:", c.image_path)
    print("  --- predicted (first 400 chars) ---\n ", c.pred_html[:400])
    print("  --- ground truth (first 400 chars) ---\n ", c.true_html[:400])


## Next: GLM-OCR

Once PaddleOCR-VL is proven end-to-end here, GLM-OCR (0.9B, MIT) gets a sibling client
`src/model/glm_client.py` with the same `predict`/`predict_many` surface, and a second
notebook (or an added cell here) that scores it over the **same** `data/corpus/eval`
records and runs `compare_runs(paddle_summary, glm_summary)` for the head-to-head.